## 摘要索引（Summary Index）

摘要索引示例代码
```
# 1.提取，分割，块（比较大）
# 2.块，生成摘要
# 3.将摘要向量化， 摘要和原始文档要建立关系
# 4.摘要向量存储到向量数据库中，原始文档存储（不是存到向量数据库）
# 5.检索，匹配摘要向量，返回原始文档
```

注意事项：
```
# 1.摘要质量至关重要: 这是一个关于deepseek的介绍文章  太过于简单，丢失了关键信息
# 2.存储开销比较大: 只存储知识块对应的向量    摘要索引：既要存储摘要的向量还要存储原始文档
# 3.一致性问题，维护一致性: 通过uuid 将原始文档和摘要一一对应
```

In [5]:
from models import get_ali_clients, get_ali_model_client  # 自定义模块，用于获取阿里云大模型和嵌入模型客户端
from langchain_ollama import OllamaEmbeddings
import warnings
warnings.filterwarnings("ignore")
# 获得访问大模型和嵌入模型客户端
# client,embeddings_model = get_ali_clients()

# 使用本地Ollama向量模型
client = get_ali_model_client()
embeddings_model = OllamaEmbeddings(model="bge-m3")

### 1. 文档加载、文档分块，每个块生成摘要

In [6]:
from langchain_community.document_loaders import TextLoader # 文本加载器，用于加载文档
from langchain_text_splitters import RecursiveCharacterTextSplitter # 递归文本分割器
from langchain_core.output_parsers import StrOutputParser   # 字符串输出解析器
from langchain_core.prompts import ChatPromptTemplate  # 聊天提示模板

# 初始化文档加载器
loader = TextLoader("../data/deepseek百度百科.txt", encoding="utf-8")

# 加载文档
docs = loader.load()

# 初始化递归文本分割器（设置块大小和重叠）
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1024, chunk_overlap=100)
docs = text_splitter.split_documents(docs)
print(f"文档分块完成，共生成 {len(docs)} 个块")

# 创建摘要生成链
chain = (
    {"doc": lambda x: x.page_content} # doc.page_content
    | ChatPromptTemplate.from_template("总结下面的文档:\n\n{doc}")
    | client
    | StrOutputParser()
)

print("准备生成文档摘要，请耐心等待...")
summaries = chain.batch(docs, {"max_concurrency": 5}) # 批量生成文档摘要（最大并发数5）
print(f"摘要生成完成，共生成 {len(summaries)} 个摘要")

文档分块完成，共生成 27 个块
准备生成文档摘要，请耐心等待...
摘要生成完成，共生成 27 个摘要


In [7]:
print(docs[0].page_content)

DeepSeek，全称杭州深度求索人工智能基础技术研究有限公司。DeepSeek是一家创新型科技公司，成立于2023年7月17日 ，使用数据蒸馏技术，得到更为精炼、有用的数据。由知名私募巨头幻方量化孕育而生，专注于开发先进的大语言模型（LLM）和相关技术。注册地址：浙江省杭州市拱墅区环城北路169号汇金国际大厦西1幢1201室。法定代表人为裴湉，经营范围包括技术服务、技术开发、软件开发等。
2024年1月5日至6月，相继发布DeepSeek LLM、DeepSeek-Coder、DeepSeekMath、DeepSeek-VL、DeepSeek-V2、DeepSeek-Coder-V2模型。 2024年9月5日，更新API支持文档，宣布合并DeepSeek Coder V2和DeepSeek V2 Chat，推出DeepSeek V2.5。12月13日，发布DeepSeek-VL2。12月26日，正式上线DeepSeek-V3首个版本并同步开源。2025年1月31日，英伟达宣布DeepSeek-R1模型登陆NVIDIANIM。同一时段内，亚马逊和微软也接入DeepSeek-R1模型。 2月5日，DeepSeek-R1、V3、Coder等系列模型，已陆续上线国家超算互联网平台。2月6日消息，澳大利亚政府以所谓“担心安全风险”为由，已禁止在所有政府设备中使用DeepSeek。2月8日，DeepSeek正式登陆苏州，并在苏州市公共算力服务平台上完成部署上线，为用户提供开箱即用的软硬件一体服务。 截至2月9日，DeepSeek App的累计下载量已超1.1亿次，周活跃用户规模最高近9700万。 2月20日消息，已有超200家企业宣布接入DeepSeek。 2月26日，DeepSeek宣布开源DeepGEMM。


In [8]:
print(summaries[0])

文档主要介绍了 DeepSeek 的公司背景与发展进展：

DeepSeek 全称是“杭州深度求索人工智能基础技术研究有限公司”，成立于 2023 年 7 月 17 日，由幻方量化孵化，是一家专注大语言模型及相关技术的创新型科技公司。公司注册地址位于杭州市拱墅区，法定代表人为裴湉，经营范围包括技术服务、技术开发和软件开发等。其技术特点之一是利用数据蒸馏技术获取更精炼、有用的数据。

在模型发展方面，DeepSeek 自 2024 年起密集发布多款模型：2024 年 1 月至 6 月相继推出 DeepSeek LLM、DeepSeek-Coder、DeepSeekMath、DeepSeek-VL、DeepSeek-V2、DeepSeek-Coder-V2；2024 年 9 月推出合并后的 DeepSeek V2.5；12 月发布 DeepSeek-VL2，并上线开源 DeepSeek-V3 首个版本。

2025 年初，DeepSeek 的生态和应用进一步扩大：DeepSeek-R1 模型登陆英伟达 NIM，并被亚马逊、微软接入；R1、V3、Coder 等系列模型上线国家超算互联网平台；同时也在苏州公共算力服务平台部署上线，提供软硬件一体服务。

不过，DeepSeek 也面临外部争议。澳大利亚政府以“安全风险”为由，禁止政府设备使用 DeepSeek。

在应用和企业采用方面，截至 2025 年 2 月 9 日，DeepSeek App 累计下载量超过 1.1 亿次，周活跃用户规模最高接近 9700 万；已有超过 200 家企业宣布接入 DeepSeek。2 月 26 日，DeepSeek 还宣布开源 DeepGEMM。


### 2. 初始化向量数据库和文档存储库

In [9]:
from langchain_classic.retrievers import MultiVectorRetriever
from langchain_chroma import Chroma
from langchain_core.stores import InMemoryByteStore
import uuid   # 用于生成唯一标识符
from langchain_core.documents import Document    # 文档对象
from langchain_core.runnables import RunnableMap, RunnableParallel  # 可运行的映射

# 初始化Chroma实例（用于存储摘要向量）
vectorstore = Chroma(
    collection_name="summaries",
    embedding_function=embeddings_model
)

# 初始化内存字节存储（用于存储原始文档）
store = InMemoryByteStore()

# 初始化多向量检索器（结合向量存储和文档存储）
id_key = "doc_id"
retriever = MultiVectorRetriever(
    vectorstore=vectorstore, # 向量检索库。存的是摘要向量，用来做相似度匹配
    byte_store=store, # 原始文档存储库。存的是原始文档chunk，通过ID关联
    id_key=id_key, # 关联键名。用于将摘要和原始文档进行关联，指定document中元数据中哪个字段作为"桥梁"，把向量库的结果映射回原文
)

# 为每个文档生成唯一ID，该ID用于关联原始文档和摘要
doc_ids = [str(uuid.uuid4()) for _ in docs]

# 将文档摘要转换为LangChain中Document
summary_docs = [
    Document(page_content=s, metadata={id_key: doc_ids[i]})
    for i, s in enumerate(summaries)
]

目前向量和文档存储都是存在本地

- 向量存储：生产环境不用Chroma，用Faiss或Milvus

- 文档存储：生产环境不用InMemoryByteStore，用MySQL或PG


数据关系图
```
vectorstore（Chroma）                    docstore（InMemoryByteStore）
┌────────────────────────┐              ┌────────────────────────┐
│ 记录1:                  │             │                        │
│   embedding: [0.02,..] │              │  "a1b2c3d4-..."        │
│   content: "摘要文本1"  │── doc_id ──→│    → docs[0] 原始文档块 │
│   metadata:            │   关联       │                        │
│     doc_id:"a1b2c3d4-" │              │  "e5f6g7h8-..."        │
├────────────────────────┤              │    → docs[1] 原始文档块 │
│ 记录2:                  │              │                        │
│   embedding: [0.15,..] │              │  "i9j0k1l2-..."        │
│   content: "摘要文本2"  │── doc_id ──→ │    → docs[2] 原始文档块 │
│   metadata:            │   关联       │                        │
│     doc_id:"e5f6g7h8-" │              │  ...                   │
├────────────────────────┤              │                        │
│ ...                    │              │                        │
└────────────────────────┘              └────────────────────────┘

```

### 3. 将`摘要向量`添加到向量数据库（Chroma），将`原始文档块`存储到内存数据库（InMemoryByteStore）

In [10]:
# 将摘要向量添加到向量数据库
print("准备将摘要添加到向量数据库...")
batch_size = 5
for i in range(0, len(summary_docs), batch_size):
    batch = summary_docs[i:i + batch_size]
    retriever.vectorstore.add_documents(batch) # 核心功能：将摘要向量添加到向量数据库，batch_size最大是20，超过20会报错，模型要求
    print(f"已添加 {min(i + batch_size, len(summary_docs))}/{len(summary_docs)} 条")

# 将原始文档存储到字节存储（使用ID关联）
print("准备将原始文档存储到字节存储...")
# mset：批量设置键值对
# list(zip(doc_ids, docs))：将ID和文档配对
retriever.docstore.mset(list(zip(doc_ids, docs)))

准备将摘要添加到向量数据库...
已添加 5/27 条
已添加 10/27 条
已添加 15/27 条
已添加 20/27 条
已添加 25/27 条
已添加 27/27 条
准备将原始文档存储到字节存储...


检索流程中的调用顺序
```
    ① 搜索                    ② 取ID              ③ 取原文
Query ──────────→ vectorstore ──────────→ doc_id ──────────→ docstore ──→ 原始文档
                  (Chroma)                (桥梁)              (ByteStore)

                  做什么：                做什么：             做什么：
                  向量相似度匹配          从metadata提取       按key精确查找
                  返回Top-K摘要          doc_id字段          返回原始chunk

```

In [11]:
# 手动测试代码 - 相似性搜索
query = "deepseek的企业事件"
sub_docs = retriever.vectorstore.similarity_search(query)
print("-------------匹配的摘要内容--------------")
print(sub_docs[0])

# 获取第一个匹配摘要的ID
matched_id = sub_docs[0].metadata[id_key]

print(f"-------------对应的原始文档--------------")
# 通过ID获取原始文档
original_doc = retriever.docstore.mget([matched_id])
print(original_doc)
# 执行相似性搜索测试---完成

-------------匹配的摘要内容--------------
page_content='文档主要介绍了 DeepSeek 的公司背景与发展进展：

DeepSeek 全称是“杭州深度求索人工智能基础技术研究有限公司”，成立于 2023 年 7 月 17 日，由幻方量化孵化，是一家专注大语言模型及相关技术的创新型科技公司。公司注册地址位于杭州市拱墅区，法定代表人为裴湉，经营范围包括技术服务、技术开发和软件开发等。其技术特点之一是利用数据蒸馏技术获取更精炼、有用的数据。

在模型发展方面，DeepSeek 自 2024 年起密集发布多款模型：2024 年 1 月至 6 月相继推出 DeepSeek LLM、DeepSeek-Coder、DeepSeekMath、DeepSeek-VL、DeepSeek-V2、DeepSeek-Coder-V2；2024 年 9 月推出合并后的 DeepSeek V2.5；12 月发布 DeepSeek-VL2，并上线开源 DeepSeek-V3 首个版本。

2025 年初，DeepSeek 的生态和应用进一步扩大：DeepSeek-R1 模型登陆英伟达 NIM，并被亚马逊、微软接入；R1、V3、Coder 等系列模型上线国家超算互联网平台；同时也在苏州公共算力服务平台部署上线，提供软硬件一体服务。

不过，DeepSeek 也面临外部争议。澳大利亚政府以“安全风险”为由，禁止政府设备使用 DeepSeek。

在应用和企业采用方面，截至 2025 年 2 月 9 日，DeepSeek App 累计下载量超过 1.1 亿次，周活跃用户规模最高接近 9700 万；已有超过 200 家企业宣布接入 DeepSeek。2 月 26 日，DeepSeek 还宣布开源 DeepGEMM。' metadata={'doc_id': '6a373ad5-006e-45ee-8eb4-aa8b4ed0c160'}
-------------对应的原始文档--------------
[Document(metadata={'source': '../data/deepseek百度百科.txt'}, page_content='DeepSeek，全称杭州深度求索人工智能基础技术研究有限公司。DeepSeek是一家创新型科技公司，成立于2023年

In [16]:
# retriever.invoke将上面对摘要进行检索，但是通过关联ID获得原始文档，最终返回原始文档的过程全部都包含完成了
retriever.invoke("deepseek的企业事件")[0]

Document(metadata={'source': '../data/deepseek百度百科.txt'}, page_content='DeepSeek，全称杭州深度求索人工智能基础技术研究有限公司。DeepSeek是一家创新型科技公司，成立于2023年7月17日 ，使用数据蒸馏技术，得到更为精炼、有用的数据。由知名私募巨头幻方量化孕育而生，专注于开发先进的大语言模型（LLM）和相关技术。注册地址：浙江省杭州市拱墅区环城北路169号汇金国际大厦西1幢1201室。法定代表人为裴湉，经营范围包括技术服务、技术开发、软件开发等。\n2024年1月5日至6月，相继发布DeepSeek LLM、DeepSeek-Coder、DeepSeekMath、DeepSeek-VL、DeepSeek-V2、DeepSeek-Coder-V2模型。 2024年9月5日，更新API支持文档，宣布合并DeepSeek Coder V2和DeepSeek V2 Chat，推出DeepSeek V2.5。12月13日，发布DeepSeek-VL2。12月26日，正式上线DeepSeek-V3首个版本并同步开源。2025年1月31日，英伟达宣布DeepSeek-R1模型登陆NVIDIANIM。同一时段内，亚马逊和微软也接入DeepSeek-R1模型。 2月5日，DeepSeek-R1、V3、Coder等系列模型，已陆续上线国家超算互联网平台。2月6日消息，澳大利亚政府以所谓“担心安全风险”为由，已禁止在所有政府设备中使用DeepSeek。2月8日，DeepSeek正式登陆苏州，并在苏州市公共算力服务平台上完成部署上线，为用户提供开箱即用的软硬件一体服务。 截至2月9日，DeepSeek App的累计下载量已超1.1亿次，周活跃用户规模最高近9700万。 2月20日消息，已有超200家企业宣布接入DeepSeek。 2月26日，DeepSeek宣布开源DeepGEMM。')

### 4. 问题回答

In [12]:
from langchain_core.output_parsers import StrOutputParser   # 字符串输出解析器
from langchain_core.prompts import ChatPromptTemplate  # 聊天提示模板

prompt =  ChatPromptTemplate.from_template("根据下面的文档回答问题:\n\n{doc}\n\n问题: {question}")
# 生成问题回答链
# RunnableMap是RunnableParallel另一种写法
# retriever.invoke将上面对摘要进行检索，但是通过关联ID获得原始文档，最终返回原始文档的过程全部都包含完成了
chain = RunnableMap({
    "doc": lambda x: retriever.invoke(x["question"]),
    "question": lambda x: x["question"]
}) | prompt | client | StrOutputParser()

# 生成问题回答
query = "deepseek的企业事件"
answer = chain.invoke({"question": query})
print("-------------回答--------------")
print(answer)
# retriever.invoke(query)  1.向量数据库中检索摘要向量   2.匹配对应的原始文档并返回
retrieved_docs = retriever.invoke(query)
print("-------------检索到的文档--------------")
print(retrieved_docs)

-------------回答--------------
根据文档内容，DeepSeek 的“企业事件”主要可以整理为以下几类：**公司成立与资本变更、模型发布与技术进展、生态合作与行业落地、开源事件、收益与经营相关事件、争议与外部事件**。

---

## 一、公司成立与早期发展事件

| 时间 | 事件 |
|---|---|
| 2023年7月17日 | DeepSeek 正式成立，全称为“杭州深度求索人工智能基础技术研究有限公司”。文档称其由知名量化资管巨头幻方量化创立/孕育而生，专注于先进大语言模型 LLM 和相关技术研发。 |
| 2023年8月2日 | 公司注册资本变更为1000万元，并完成章程备案；投资人变更为宁波程恩企业管理咨询合伙企业（有限合伙）；市场主体类型变更为其他有限责任公司。 |

---

## 二、模型发布与技术进展事件

| 时间 | 事件 |
|---|---|
| 2024年1月5日至6月 | DeepSeek 相继发布多个模型，包括 DeepSeek LLM、DeepSeek-Coder、DeepSeekMath、DeepSeek-VL、DeepSeek-V2、DeepSeek-Coder-V2。 |
| 2024年5月7日 | 发布 MoE 架构的 DeepSeek V2。 |
| 2024年9月5日 | DeepSeek 更新 API 支持文档，宣布合并 DeepSeek Coder V2 和 DeepSeek V2 Chat，推出 DeepSeek V2.5。API 用户可通过 `deepseek-coder` 或 `deepseek-chat` 访问新模型。 |
| 2024年12月13日 | 发布 DeepSeek-VL2。 |
| 2024年12月26日 | 正式上线 DeepSeek-V3 首个版本，并同步开源。 |
| 2025年1月31日 | 英伟达宣布 DeepSeek-R1 模型登陆 NVIDIA NIM。同一时期，亚马逊和微软也接入 DeepSeek-R1 模型。 |
| 2025年2月5日 | DeepSeek-R1、V3、Coder 等系列模型上线国家超算互联网平台。 |
| 2025年2月21日 | DeepSeek 团队宣布，将从下周开始陆续开源 5 个代码库，并将该计划称为 “Open S